Name: John Laine
Dataset Selected: Titanic - Machine Learning from Disaster

TODO: Create a short 2–3 sentence description of the project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("./data/titanic/train.csv")
df.head()

## Data Cleaning

The raw Titanic dataset has several data quality issues that must be addressed before analysis:

- `Age` is missing for roughly 20% of passengers.
- `Cabin` is missing for roughly 77% of passengers — too sparse to use directly, but the presence/absence of a cabin record is itself informative.
- `Embarked` is missing for a small number of rows.
- `Sex` and `Embarked` are stored as plain strings; converting them to categoricals and expanding the port codes improves readability and downstream modeling.

The two functions below address these issues. Each is written to take a DataFrame and return a cleaned copy (no in-place mutation), so the original `df` remains available for comparison.

In [ ]:
def handle_missing_values(df):
    """Return a copy of the Titanic DataFrame with missing values handled.

    Cleaning steps:
        * ``Age``      — imputed with the median age within each
                         (Pclass, Sex) group, since age varies meaningfully
                         by passenger class and sex. Falls back to the
                         overall median for any remaining gaps.
        * ``Embarked`` — imputed with the most common port (mode), since
                         only a handful of rows are missing.
        * ``Cabin``    — too sparse to impute (~77% missing). Replaced with
                         a binary ``HasCabin`` indicator (1 if a cabin was
                         recorded, 0 otherwise), and the original column
                         is dropped.

    Args:
        df: Raw Titanic DataFrame as loaded from ``train.csv``.

    Returns:
        A new DataFrame with no missing values in ``Age`` or ``Embarked``,
        a new ``HasCabin`` column, and ``Cabin`` removed.
    """
    df = df.copy()

    df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
        lambda s: s.fillna(s.median())
    )
    df["Age"] = df["Age"].fillna(df["Age"].median())

    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df = df.drop(columns=["Cabin"])

    return df

In [ ]:
def standardize_features(df):
    """Return a copy of the DataFrame with standardized types and engineered features.

    Transformations:
        * ``Sex``       — cast to a ``category`` dtype for clarity and lower
                          memory usage.
        * ``Embarked``  — port codes (``C``, ``Q``, ``S``) are expanded to
                          their full names (``Cherbourg``, ``Queenstown``,
                          ``Southampton``) and cast to ``category``.
        * ``Pclass``    — cast to an ordered ``category`` (1st > 2nd > 3rd)
                          so plots and groupings preserve class order.
        * ``FamilySize`` — engineered feature equal to
                           ``SibSp + Parch + 1`` (siblings/spouses plus
                           parents/children plus the passenger themself).
        * ``IsAlone``   — engineered binary flag, 1 when ``FamilySize == 1``.

    Args:
        df: A Titanic DataFrame, typically the output of
            :func:`handle_missing_values`.

    Returns:
        A new DataFrame with standardized categorical columns and two
        additional engineered features.
    """
    df = df.copy()

    df["Sex"] = df["Sex"].astype("category")

    embark_map = {"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"}
    df["Embarked"] = df["Embarked"].map(embark_map).astype("category")

    df["Pclass"] = pd.Categorical(df["Pclass"], categories=[1, 2, 3], ordered=True)

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    return df

### Apply the cleaning functions

Below the functions are applied in sequence to produce a cleaned DataFrame `df_clean`. A quick check of missing-value counts and dtypes before and after confirms the transformations took effect.

In [ ]:
print("Missing values BEFORE cleaning:")
print(df.isna().sum()[df.isna().sum() > 0])

df_clean = standardize_features(handle_missing_values(df))

print("\nMissing values AFTER cleaning:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0] if df_clean.isna().any().any() else "None")

print("\nDtypes after cleaning:")
print(df_clean.dtypes)

df_clean.head()

## Exploratory Data Analysis

With the dataset cleaned, the next step is exploratory analysis. The function below produces a structured snapshot of any DataFrame — shape, dtypes, summary statistics, and survival-rate breakdowns by key categorical features — and a numeric correlation matrix. Packaging this as a function makes the EDA reusable on future slices of the data (e.g., a subset filtered to first-class passengers) without duplicating code.

In [ ]:
def explore_dataset(df, target="Survived", group_cols=("Sex", "Pclass", "Embarked", "IsAlone")):
    """Generate a structured exploratory summary of a Titanic-style DataFrame.

    For the supplied DataFrame, this function prints:
        1. Shape (rows, columns) and column dtypes.
        2. ``describe()`` for numeric columns (count, mean, std, quartiles).
        3. Mean of the ``target`` column grouped by each column in
           ``group_cols`` — for the Titanic dataset, this surfaces the
           survival rate within each category (e.g. survival rate by sex,
           by passenger class, etc.).
        4. A Pearson correlation matrix of all numeric columns, useful for
           spotting features that move together with the target.

    Args:
        df: The DataFrame to analyze.
        target: Name of the numeric/binary outcome column to break down by
            each grouping (default ``"Survived"``).
        group_cols: Iterable of categorical columns to use as groupings
            for the target-mean breakdown. Columns not present in ``df``
            are skipped silently so the function works on filtered slices.

    Returns:
        The Pearson correlation matrix (a DataFrame), so the caller can
        reuse it (e.g. to plot a heatmap) without recomputing.
    """
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

    print("Dtypes:")
    print(df.dtypes, "\n")

    print("Numeric summary statistics:")
    print(df.describe(), "\n")

    if target in df.columns:
        for col in group_cols:
            if col in df.columns:
                print(f"Mean {target} by {col}:")
                print(df.groupby(col, observed=True)[target].mean().round(3), "\n")

    numeric_df = df.select_dtypes(include="number")
    corr = numeric_df.corr()
    print("Correlation matrix (numeric columns):")
    print(corr.round(2))

    return corr

In [ ]:
corr = explore_dataset(df_clean)